# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walk-through for downloading, exploring, and analyzing the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: 
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed (uncomment in Colab/first run)
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and display information
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review record sets, fields, and their `@id`s, *using `@id` for referencing entities!*

In [ ]:
# List all record sets and their IDs
record_sets = [rs['@id'] for rs in dataset.metadata_json.get('recordSet', [])]
if not record_sets:
    print('⚠️ No record sets were defined in top-level metadata. Attempting to infer from files...')
    # Fallback: Try to list distributions and infer record sets
    dists = dataset.metadata_json.get('distribution', [])
    for dist in dists:
        print(f"Distribution @id: {dist['@id']}")
    print("\nYou may need to inspect the Croissant metadata manually.")
else:
    for rs in dataset.metadata_json['recordSet']:
        rsid = rs['@id']
        print(f"RecordSet @id: {rsid}")
        if 'field' in rs:
            fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
            print("  Fields:")
            for field in fields:
                print(f"    - {field['@id']}: {field.get('name','')} ({field.get('dataType','')})")
        print()

We'll use the dataset's main record set containing protein information. According to [the Croissant file](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json), the main tabular data is in a distribution whose record set `@id` is typically like `_:main_record_set`—let's autodetect it.

In [ ]:
# Helper: List the internal Croissant JSON for record sets, fields, columns
import pprint
print("Available record sets (extracted from Croissant JSON):")
record_sets_json = dataset.metadata_json.get('recordSet', [])
for rs in record_sets_json:
    pprint.pprint({k: v for k, v in rs.items() if k in ['@id', 'name', 'field', 'dataType', 'source']})

# Pick the main record set (first found, for this dataset)
if record_sets_json:
    main_record_set_id = record_sets_json[0]['@id']
    print(f"\nMain record set @id: {main_record_set_id}")
else:
    # Sometimes inline files/distribution is the only source. We'll guess.
    main_record_set_id = None
    print('No record sets found; you may need to download and inspect files directly.')

## 3. Data Extraction
Load data from the selected record set (`@id` indicated above) into a dataframe for analysis. If there are multiple record sets, extract all into dataframes using their `@id` as keys.

In [ ]:
# List all available record set ids
record_set_ids = [rs['@id'] for rs in record_sets_json] if record_sets_json else []
dataframes = {}

for rsid in record_set_ids:
    print(f"Loading records for record set: {rsid}")
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            df = pd.DataFrame(records)
            dataframes[rsid] = df
            print(f"  Columns: {list(df.columns)[:10]}{'...' if df.shape[1]>10 else ''}")
            print(f"  Number of records: {df.shape[0]}")
        else:
            print("  (No data records found)")
    except Exception as e:
        print(f"  Error loading records: {e}")

# Pick a record set with tabular data for EDA
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]
    print(f"\nChosen record set for analysis: {chosen_record_set_id}")
    print("Fields (columns) available:")
    print(list(dataframes[chosen_record_set_id].columns))
    display(dataframes[chosen_record_set_id].head())
else:
    print("⚠️ No tabular dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (such as peptide counts, abundance, or coverage percentage as available in the record set) for filtering, normalization, and grouping.

> **Note**: You can view all column field `@id`s with `print(dataframes[chosen_record_set_id].columns.tolist())`.

In [ ]:
# Pick a numeric field (using the Croissant field @id or column name)
df = dataframes[chosen_record_set_id]
print("Candidate numeric columns:")
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        print(col)

# Suggest likely numeric fields (by heuristic)
likely_numeric = [c for c in df.columns if
                  any(sub in c.lower() for sub in [
                      'abundance', 'count', 'coverage', 'mw', 'weight', 'pi', 'number', 'percent', 'sequence_length', 'score'
                  ]) and pd.api.types.is_numeric_dtype(df[c])]
if likely_numeric:
    numeric_field = likely_numeric[0]
    print(f"Selected numeric field for EDA: {numeric_field}")
else:
    numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    numeric_field = numeric_cols[0] if numeric_cols else df.columns[0]
    print(f"No obvious numeric fields found by name. Falling back to: {numeric_field}")

# Threshold filter (edit this if your field has a different range)
threshold = df[numeric_field].quantile(0.75)
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records where {numeric_field} > {threshold:.2f}: {filtered_df.shape[0]} records out of {df.shape[0]}")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Attempt grouping by a categorical field (e.g. sample, accession, protein name, ...)
categorical_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) or df[col].dtype.name=='category']
# Heuristically pick a field for grouping
group_field = None
for cat in ['sample', 'condition', 'accession', 'protein', 'group', 'description']:
    for c in categorical_candidates:
        if cat in c.lower():
            group_field = c
            break
    if group_field:
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False)
    print(f"Top groups by mean {numeric_field} (grouped by '{group_field}'):")
    display(grouped_df.head(10))
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Let's plot the distribution of the selected numeric field and group means (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Distribution plot
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=30)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If we have grouped means, plot top 10
if 'grouped_df' in locals() and group_field:
    plt.figure(figsize=(8,4))
    grouped_df.head(10).plot(kind='bar')
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- This notebook demonstrated how to use `mlcroissant` to load and inspect mass spectrometry data from extracellular vesicle isolation experiments.
- The exploration included summary statistics, field normalization, and simple grouping of protein features.
- For more detailed analyses or RAI exploration, consult the Croissant schema fields or original dataset publication for scientific questions, or modify the code cells to answer your research objectives.

*Notebook prepared using [mlcroissant](https://github.com/mlcommons/croissant) with Croissant schema at*  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`